In [5]:
# Terminal e 'pip install ultralytics' na kora thakle VS Code er terminal theke kore niben.
import os

# Apni already VS Code e 'blood_cell_project' folder open kore rakhechen, 
# tai amra directly current working directory tai use korbo.
PROJECT_ROOT = os.getcwd()
print(f"Working Directory: {PROJECT_ROOT}")

Working Directory: /home/linux-hasan/projects/blood_cell_project


In [6]:
import json
import os

def coco_to_yolo(json_path, output_label_dir):
    os.makedirs(output_label_dir, exist_ok=True)

    with open(json_path, 'r') as f:
        data = json.load(f)

    images = {img['id']: img for img in data['images']}

    # Clear existing files to prevent duplicate appending
    for f_name in os.listdir(output_label_dir):
        os.remove(os.path.join(output_label_dir, f_name))

    for ann in data['annotations']:
        img_info = images[ann['image_id']]
        img_w, img_h = img_info['width'], img_info['height']

        # COCO to YOLO math
        x_min, y_min, w, h = ann['bbox']
        x_center = (x_min + w / 2) / img_w
        y_center = (y_min + h / 2) / img_h
        norm_w = w / img_w
        norm_h = h / img_h

        cat_id = ann['category_id']
        line = f"{cat_id} {x_center} {y_center} {norm_w} {norm_h}\n"

        txt_name = os.path.splitext(img_info['file_name'])[0] + '.txt'
        txt_path = os.path.join(output_label_dir, txt_name)

        with open(txt_path, 'a') as out_f:
            out_f.write(line)

# Setup paths based on your merged_pretrain structure
PRETRAIN_DIR = os.path.join(PROJECT_ROOT, "2_preprocessed_datasets", "merged_pretrain")

train_json = os.path.join(PRETRAIN_DIR, "annotations", "train.json")
val_json = os.path.join(PRETRAIN_DIR, "annotations", "val.json")

train_labels = os.path.join(PRETRAIN_DIR, "labels", "train")
val_labels = os.path.join(PRETRAIN_DIR, "labels", "val")

print("Converting Train JSON to YOLO txt format...")
coco_to_yolo(train_json, train_labels)

print("Converting Val JSON to YOLO txt format...")
coco_to_yolo(val_json, val_labels)

print("✅ Conversion successful. Labels are ready.")

Converting Train JSON to YOLO txt format...
Converting Val JSON to YOLO txt format...
✅ Conversion successful. Labels are ready.


In [7]:
dataset_yaml_path = os.path.join(PROJECT_ROOT, "scripts", "ultralytics_blood_cell.yaml")

yaml_content = f"""
path: {PRETRAIN_DIR}
train: images/train
val: images/val

# Classes (Based on your unified mapping)
names:
  0: RBC
  1: WBC
  2: Platelet
"""

os.makedirs(os.path.dirname(dataset_yaml_path), exist_ok=True)
with open(dataset_yaml_path, "w") as f:
    f.write(yaml_content)

print(f"✅ Data config created exactly at: {dataset_yaml_path}")

✅ Data config created exactly at: /home/linux-hasan/projects/blood_cell_project/scripts/ultralytics_blood_cell.yaml


In [8]:
from ultralytics import RTDETR
import os

print("🚀 Starting Stage 1 Pretraining with Official Weights...")

# Load the official pre-trained RT-DETR-Large model from Ultralytics
model = RTDETR('rtdetr-l.pt')

# Dynamically set output directory
output_dir = os.path.join(PROJECT_ROOT, "3_models")

# Train the model
results = model.train(
    data=os.path.join(PROJECT_ROOT, "scripts", "ultralytics_blood_cell.yaml"),
    epochs=150,        # Good for pretraining
    patience=20,       # Halts early if no improvement
    imgsz=640,
    batch=8,           # Local GPU thakle 8 ba 16 use korte paren. Error ashle 4 e fire jaben.
    workers=4,         # Local CPU er jonno 4 ba 8 use korle data loading fast hobe
    project=output_dir,
    name='stage1_pretrained',
    device=0,
    amp=True
)

print("✅ Pretraining completed successfully!")

🚀 Starting Stage 1 Pretraining with Official Weights...


100%|██████████| 63.4M/63.4M [00:07<00:00, 9.32MB/s]


New https://pypi.org/project/ultralytics/8.4.22 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/linux-hasan/projects/blood_cell_project/scripts/ultralytics_blood_cell.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi

100%|██████████| 5.35M/5.35M [00:00<00:00, 9.57MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 167.3±111.9 MB/s, size: 63.8 KB)


...

  0%|          | 0/148 [00:00<?, ?it/s]/home/linux-hasan/projects/envs/pt/lib/python3.12/site-packages/torch/autograd/graph.py:824: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:93.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
     97/150       6.9G     0.1648     0.3235    0.07878        107        640: 100%|██████████| 148/148 [00:50<00:00,  2.92it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:02<00:00,  6.76it/s]

                   all        312       4549      0.775      0.812      0.812      0.676
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 77, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



97 epochs completed in 1.447 hours.
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/stage1_pretrained/weights/last.pt, 66.2MB
Optimizer stripped from /home/linux-hasan/projects/blood_cell_project/3_models/stage1_pretrained/weights/best.pt, 66.2MB

Validating /home/linux-hasan/projects/blood_cell_project/3_models/stage1_pretrained/weights/best.pt...
Ultralytics 8.3.166 🚀 Python-3.12.3 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070 SUPER, 12282MiB)
rt-detr-l summary: 302 layers, 31,989,905 parameters, 0 gradients, 103.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 20/20 [00:03<00:00,  5.53it/s]


                   all        312       4549      0.783      0.774      0.839      0.697
                   RBC        307        957      0.704      0.633      0.725        0.6
                   WBC        312       3431      0.772      0.814      0.859      0.789
              Platelet         99        161      0.872      0.876      0.933      0.702
Speed: 0.1ms preprocess, 6.8ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/linux-hasan/projects/blood_cell_project/3_models/stage1_pretrained
✅ Pretraining completed successfully!
